In [0]:
%run ./00_project_setup

# AI Workforce Capacity Planning Platform
## Notebook 07 — Enterprise Metadata Management

**Implementation:** 08 — Enterprise Metadata Management  
**Implementation version:** 1.0.0

This notebook validates the enterprise metadata subsystem, including:

1. metadata domain models,
2. Spark dataset profiling,
3. deterministic dataset fingerprinting,
4. catalog persistence and retrieval,
5. metadata service orchestration,
6. an end-to-end metadata registration workflow.

## Section 01 — Metadata Domain Validation

In [0]:
from src.metadata import (
    ColumnProfile,
    DatasetFingerprint,
    DatasetProfile,
    DatasetStatistics,
    MetadataCatalogEntry,
)

print("Implementation 07 metadata domain imports: PASSED")

## Section 02 — Spark Dataset Profiler Validation

This section validates the reusable Spark profiling engine using a controlled
dataset with known row counts, duplicate rows, null values, numeric fields,
string fields, and a date field.

In [0]:
from pyspark.sql import functions as F

from src.metadata import SparkDatasetProfiler


profiler_test_df = (
    spark.createDataFrame(
        [
            (1, "Monday", 10000.0, "2026-07-27"),
            (2, "Tuesday", 10500.0, "2026-07-28"),
            (3, "Wednesday", None, "2026-07-29"),
            (3, "Wednesday", None, "2026-07-29"),
        ],
        [
            "record_id",
            "day_name",
            "released_lines",
            "release_date_text",
        ],
    )
    .withColumn(
        "release_date",
        F.to_date("release_date_text"),
    )
    .drop("release_date_text")
)

metadata_profiler = SparkDatasetProfiler(
    approximate_distinct=False
)

dataset_statistics, column_profiles = metadata_profiler.profile(
    profiler_test_df
)

print("Row count:", dataset_statistics.row_count)
print("Column count:", dataset_statistics.column_count)
print("Duplicate rows:", dataset_statistics.duplicate_row_count)
print("Null cells:", dataset_statistics.null_cell_count)
print("Columns profiled:", len(column_profiles))

assert dataset_statistics.row_count == 4
assert dataset_statistics.column_count == 4
assert dataset_statistics.duplicate_row_count == 1
assert dataset_statistics.null_cell_count == 2
assert len(column_profiles) == 4

print("Implementation 07 Spark dataset profiler: PASSED")

## Section 03 — Dataset Fingerprint Validation 

Validate deterministic schema, content, metadata, and combined dataset  
fingerprints using the profiler test dataset.

In [0]:
from src.metadata import (
    DatasetFingerprint,
    DatasetFingerprintGenerator,
)


fingerprint_generator = DatasetFingerprintGenerator()

fingerprint_metadata = {
    "dataset_key": "metadata_profiler_test",
    "dataset_layer": "TEST",
    "project_name": PROJECT_NAME,
    "environment": ENVIRONMENT,
}


first_fingerprint = fingerprint_generator.generate(
    profiler_test_df,
    metadata=fingerprint_metadata,
    statistics=dataset_statistics,
)

second_fingerprint = fingerprint_generator.generate(
    profiler_test_df,
    metadata=fingerprint_metadata,
    statistics=dataset_statistics,
)


print("Schema hash        :", first_fingerprint.schema_hash)
print("Content hash       :", first_fingerprint.content_hash)
print("Metadata hash      :", first_fingerprint.metadata_hash)
print("Combined hash      :", first_fingerprint.combined_hash)
print("Row count          :", first_fingerprint.row_count)
print("Column count       :", first_fingerprint.column_count)
print("Fingerprint version:", first_fingerprint.fingerprint_version)
print("Algorithm          :", first_fingerprint.algorithm)
print("Generated at UTC   :", first_fingerprint.generated_at_utc)


assert isinstance(
    first_fingerprint,
    DatasetFingerprint,
)

assert first_fingerprint.row_count == 4
assert first_fingerprint.column_count == 4

assert len(first_fingerprint.schema_hash) == 64
assert len(first_fingerprint.content_hash) == 64
assert len(first_fingerprint.metadata_hash) == 64
assert len(first_fingerprint.combined_hash) == 64

assert first_fingerprint.fingerprint_version == "1.0.0"
assert first_fingerprint.algorithm == "SHA-256"

assert (
    first_fingerprint.schema_hash
    == second_fingerprint.schema_hash
)

assert (
    first_fingerprint.content_hash
    == second_fingerprint.content_hash
)

assert (
    first_fingerprint.metadata_hash
    == second_fingerprint.metadata_hash
)

assert (
    first_fingerprint.combined_hash
    == second_fingerprint.combined_hash
)

assert DatasetFingerprintGenerator.fingerprints_match(
    first_fingerprint,
    second_fingerprint,
)

assert (
    first_fingerprint.generated_at_utc
    <= second_fingerprint.generated_at_utc
)

print(
    "Implementation 07 dataset fingerprinting: PASSED"
)

## Section 04 — End-to-End Metadata Workflow

# COMMAND ----------
# MAGIC %md
# MAGIC 4.1 Create Sample Dataset

In [0]:
sample_df = spark.createDataFrame(
    [
        (1001, "Alice", "Engineering", 85000.0),
        (1002, "Bob", "Operations", 72000.0),
        (1003, "Charlie", "Finance", 91000.0),
        (1004, "Diana", "Engineering", 88000.0),
    ],
    [
        "employee_id",
        "employee_name",
        "department",
        "salary",
    ],
)

display(sample_df)

# COMMAND ----------
# MAGIC %md
# MAGIC 4.2 Initialize Metadata Service

In [0]:
from src.metadata.service import MetadataService

metadata_service = MetadataService(
    spark=spark,
    catalog_path=METADATA_CATALOG_PATH,
)

print(f"Metadata catalog path: {METADATA_CATALOG_PATH}")

# COMMAND ----------
# MAGIC %md
# MAGIC 4.3 Register Dataset

In [0]:
import uuid

execution_id = str(uuid.uuid4())

dataset_profile, catalog_entry = metadata_service.register_dataset(
    dataframe=sample_df,
    dataset_name="Employee Sample Dataset",
    dataset_key="employee_sample_dataset",
    layer="silver",
    storage_path=SAMPLE_DATASET_PATH,
    storage_format="delta",
    execution_id=execution_id,
    pipeline_name="enterprise-metadata-demo",
    pipeline_version="1.0.0",
    owner="Data Engineering",
    business_description=(
        "Sample dataset demonstrating enterprise metadata registration."
    ),
    overwrite=True,
)

# COMMAND ----------
# MAGIC %md
# MAGIC 4.4 Dataset Profile

In [0]:
print(dataset_profile)

# COMMAND ----------
# MAGIC %md
# MAGIC 4.5 Metadata Catalog Entry

In [0]:
print(catalog_entry)

# COMMAND ----------
# MAGIC %md
# MAGIC 4.6 Verify Registration

In [0]:
print(
    "Dataset Exists:",
    metadata_service.dataset_exists("employee_sample_dataset"),
)

registered_dataset = metadata_service.get_dataset(
    "employee_sample_dataset"
)

print(registered_dataset)

print(
    "Catalog Size:",
    metadata_service.count_datasets(),
)

# COMMAND ----------
# MAGIC %md
# MAGIC 4.7 Metadata Catalog

In [0]:
catalog_df = metadata_service.catalog_dataframe()

display(catalog_df)


## Section 05 — Execution Summary

In [0]:
print("=" * 70)
print("IMPLEMENTATION 08 - ENTERPRISE METADATA MANAGEMENT")
print("=" * 70)

print("✓ Dataset Registration ............... PASSED")
print("✓ Dataset Profiling .................. PASSED")
print("✓ Dataset Fingerprinting ............. PASSED")
print("✓ Metadata Catalog ................... PASSED")
print("✓ Metadata Service ................... PASSED")
print("✓ End-to-End Workflow ................ PASSED")

print("-" * 70)

print(f"Dataset Name        : {catalog_entry.dataset_name}")
print(f"Dataset Key         : {catalog_entry.dataset_key}")
print(f"Layer               : {catalog_entry.layer}")
print(f"Storage Format      : {catalog_entry.storage_format}")
print(f"Row Count           : {catalog_entry.row_count}")
print(f"Column Count        : {catalog_entry.column_count}")

print("-" * 70)

print("Implementation      : 08")
print("Version             : 1.0.0")
print("Status              : PASSED")
print("Next                : Implementation 09")

print("=" * 70)